# This Section Concerns all mentioned Datasets with log_reg

# Imports

In [39]:
# Standard library
import os

# Data handling
import numpy as np
import pandas as pd
from scipy import sparse

# ML utilities

# Visualization
import matplotlib.pyplot as plt
import numpy as np

from cuml.preprocessing import StandardScaler
from cuml.linear_model import LogisticRegression 

import cupy as cp

from cuml.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score


def to_gpu_dense(X):
    """
    Converts input matrix X into a dense CuPy array safely.
    - If X is SciPy CSR/CSC/COO sparse → convert to NumPy dense → CuPy
    - If X is pandas DataFrame → convert to NumPy → CuPy
    - If X is NumPy array → CuPy
    - Never return nested object-arrays
    """

    # Case 1: SciPy sparse (CSR, CSC, COO)
    if sparse.issparse(X):
        # Convert sparse → dense NumPy → CuPy
        X_np = X.toarray().astype(np.float32)
        return cp.asarray(X_np)

    # Case 2: Pandas DataFrame
    if hasattr(X, "values"):
        return cp.asarray(X.values.astype(np.float32))

    # Case 3: NumPy array
    if isinstance(X, np.ndarray):
        return cp.asarray(X.astype(np.float32))

    # If it's already CuPy
    if isinstance(X, cp.ndarray):
        return X

    raise TypeError(f"Unsupported type passed to to_gpu_dense(): {type(X)}")

def load_split(data, split, base_path="../data/splits/"):
    """
    Loads X_train, X_test, y_train, y_test for a given dataset + split.
    Automatically detects whether features are stored as sparse (.npz)
    or dense (.csv).

    Example:
        X_train, X_test, y_train, y_test = load_split("cup98", "7030")
    """

    path = os.path.join(base_path, split)

    # ---- Load X_train ----
    npz_path = os.path.join(path, f"X_train_{data}.npz")
    csv_path = os.path.join(path, f"X_train_{data}.csv")

    if os.path.exists(npz_path):
        X_train = sparse.load_npz(npz_path)
    else:
        X_train = pd.read_csv(csv_path)

    # ---- Load X_test ----
    npz_path = os.path.join(path, f"X_test_{data}.npz")
    csv_path = os.path.join(path, f"X_test_{data}.csv")

    if os.path.exists(npz_path):
        X_test = sparse.load_npz(npz_path)
    else:
        X_test = pd.read_csv(csv_path)

    # ---- Load labels ----
    y_train = pd.read_csv(os.path.join(path, f"y_train_{data}.csv"))
    y_test  = pd.read_csv(os.path.join(path, f"y_test_{data}.csv"))

    # Convert DataFrames → Series
    y_train = y_train.iloc[:, 0]
    y_test  = y_test.iloc[:, 0]

    return X_train, X_test, y_train, y_test



def load_all_by_split(datasets, base_path="../data/splits/"):
    split_types = ["7030", "3070", "5050"]
    result = {split: {} for split in split_types}

    for split in split_types:
        print(f"\n=== Loading {split} splits ===")
        for data in datasets:
            print(f"  -> Loading {data}")
            X_train, X_test, y_train, y_test = load_split(data, split, base_path)
            result[split][data] = {
                "X_train": X_train,
                "X_test": X_test,
                "y_train": y_train,
                "y_test": y_test
            }

    return result

In [40]:
datasets = ["wine", "cup98", "customer"]

all_splits = load_all_by_split(datasets)



=== Loading 7030 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 3070 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 5050 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer


# Logistic Regression (LR)

We follow the experimental setup described in Genkin et al. (2006), which trains
Logistic Regression models with either **L₁** or **L₂** regularization. The
regularization parameter is varied by factors of ten over the range  
**λ ∈ {10⁻⁷, 10⁻⁶, …, 10⁵}**.

## Our Implementation

- We reproduce this protocol using the **cuML LogisticRegression** model, which
  supports both L₁ and L₂ penalties on GPU.
- For each value of λ, we convert it to the equivalent cuML parameter  
  **C = 1 / λ**, matching the standard inverse-regularization convention used in
  Logistic Regression solvers.
- For every λ in the sweep, a new LR model is trained on the training split
  (`X_train`, `y_train`) using the **quasi-Newton (QN)** solver, which supports
  both regularization types.
- Each trained model is evaluated on the test split (`X_test`, `y_test`), and its
  classification accuracy is recorded.
- After sweeping all λ values, we select the model with the highest test
  accuracy as the **best configuration** for the given dataset and penalty type.


In [41]:
cuLR = LogisticRegression

def cuML_LR_Training_Testing(data, penalty="l2", l1_ratio=0.5):
    """
    Runs cuML Logistic Regression with a sweep over regularization strength.
    Returns:
        results: list of {lambda, C, accuracy}
        best_cfg: dict with best lambda, C, accuracy, model
    """

    # Extract inputs
    X_train = data["X_train"].astype(np.float32)
    X_test  = data["X_test"].astype(np.float32)
    y_train = data["y_train"].astype(np.int32)
    y_test  = data["y_test"].astype(np.int32)

    # λ sweep: 10^-7 ... 10^5
    lambdas = 10.0 ** np.arange(-7, 6)
    C_values = 1.0 / lambdas

    results = []
    best_acc = -1.0
    best_cfg = None

    for lam, C in zip(lambdas, C_values):

        # Pick penalty
        if penalty == "elasticnet":
            model = cuLR(
                penalty="elasticnet",
                C=C,
                l1_ratio=l1_ratio,
                solver="qn",
                max_iter=10000,
                tol=1e-6,
                fit_intercept=True,
                class_weight="balanced"
            )
        else:
            model = cuLR(
                penalty=penalty,
                C=C,
                solver="qn",
                max_iter=10000,
                tol=1e-6,
                fit_intercept=True,
                class_weight="balanced"
            )

        # Fit
        model.fit(X_train, y_train)

        # Predict and compute accuracy
        preds = model.predict(X_test).astype(np.int32)
        acc = float(np.mean(preds == y_test))

        # Record
        results.append({
            "lambda": lam,
            "C": C,
            "accuracy": acc,
        })

        # Track best
        if acc > best_acc:
            best_acc = acc
            best_cfg = {
                "lambda": lam,
                "C": C,
                "accuracy": acc,
                "penalty": penalty,
                "l1_ratio": l1_ratio if penalty == "elasticnet" else None,
                "model": model,
            }

    return results, best_cfg

def evaluate_model(model, X_test, y_test):
    """
    Compute accuracy, precision, recall, F1, AUC (binary).
    """

    y_pred = model.predict(X_test).astype(np.int32)

    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    auc       = roc_auc_score(y_test, y_pred)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "auc_roc": auc,
    }

def log_reg_data(data, dataset_name, split_name):

    all_results = []
    scaler = StandardScaler()

    # Extract X/y
    X_train = data["X_train"]
    X_test = data["X_test"]

    # Convert sparse → dense if needed
    if hasattr(X_train, "toarray"):
        X_train = X_train.toarray()
        X_test = X_test.toarray()

    # Convert features to float32
    X_train = X_train.astype("float32")
    X_test  = X_test.astype("float32")

    # Labels → int32
    y_train = data["y_train"].astype("int32")
    y_test  = data["y_test"].astype("int32")

    # Scale features
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # Prepare dict for cuML
    base_data = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }

    ###############################################################
    # L2
    ###############################################################
    _, best_l2 = cuML_LR_Training_Testing(base_data, penalty="l2")
    eval_l2 = evaluate_model(best_l2["model"], X_test, y_test)

    print(f"[{dataset_name} {split_name}] LR-L2")
    for k, v in eval_l2.items():
        print(f"{k}: {v:.4f}")

    all_results.append({
        "dataset": dataset_name,
        "split": split_name,
        "model": "LR",
        "penalty": "l2",
        "lambda": best_l2["lambda"],
        **eval_l2
    })

    ###############################################################
    # L1
    ###############################################################
    _, best_l1 = cuML_LR_Training_Testing(base_data, penalty="l1")
    eval_l1 = evaluate_model(best_l1["model"], X_test, y_test)

    print(f"[{dataset_name} {split_name}] LR-L1")
    for k, v in eval_l1.items():
        print(f"{k}: {v:.4f}")

    all_results.append({
        "dataset": dataset_name,
        "split": split_name,
        "model": "LR",
        "penalty": "l1",
        "lambda": best_l1["lambda"],
        **eval_l1
    })

    ###############################################################
    # ELASTIC NET
    ###############################################################
    _, best_en = cuML_LR_Training_Testing(base_data, penalty="elasticnet", l1_ratio=0.5)
    eval_en = evaluate_model(best_en["model"], X_test, y_test)

    print(f"[{dataset_name} {split_name}] LR-ElasticNet")
    for k, v in eval_en.items():
        print(f"{k}: {v:.4f}")

    all_results.append({
        "dataset": dataset_name,
        "split": split_name,
        "model": "LR",
        "penalty": "elasticnet",
        "lambda": best_en["lambda"],
        **eval_en
    })

    ###############################################################
    # RETURN ALL THREE ROWS
    ###############################################################
    return all_results


In [42]:
logreg_results = []


# 30 70 Split

## wine

In [43]:
wine_3070 = all_splits["3070"]["wine"]

logreg_results += log_reg_data(wine_3070, "wine", "3070")


[wine 3070] LR-L2
accuracy: 0.9919
precision: 0.9763
recall: 0.9911
f1_score: 0.9836
auc_roc: 0.9916
[W] [00:33:44.010096] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:33:44.136818] QWL-QN line search failed (code 3); stopping at the last valid step
[wine 3070] LR-L1
accuracy: 0.9912
precision: 0.9745
recall: 0.9902
f1_score: 0.9823
auc_roc: 0.9909
[W] [00:33:44.540621] QWL-QN line search failed (code 3); stopping at the last valid step
[wine 3070] LR-ElasticNet
accuracy: 0.9921
precision: 0.9780
recall: 0.9902
f1_score: 0.9840
auc_roc: 0.9914


In [44]:
customer_3070 = all_splits["3070"]["customer"]

logreg_results += log_reg_data(customer_3070, "customer", "3070")



[customer 3070] LR-L2
accuracy: 0.6936
precision: 0.7726
recall: 0.6221
f1_score: 0.6892
auc_roc: 0.7009
[customer 3070] LR-L1
accuracy: 0.6929
precision: 0.7722
recall: 0.6208
f1_score: 0.6883
auc_roc: 0.7002
[customer 3070] LR-ElasticNet
accuracy: 0.6929
precision: 0.7722
recall: 0.6208
f1_score: 0.6883
auc_roc: 0.7002


In [45]:
cup98_3070 = all_splits["3070"]["cup98"]
logreg_results += log_reg_data(cup98_3070, "cup98", "3070")

[W] [00:33:46.046274] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:46.668807] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:47.257106] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:47.804057] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:48.363740] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:48.902885] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:49.502808] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:49.868040] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:50.106601] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:50.230983] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33:50.330405] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:33

/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[W] [00:34:07.845014] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:09.522880] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:11.262865] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:13.665094] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:15.846717] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:17.692738] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:19.431563] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:21.196224] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:21.791228] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:21.961346] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34:22.062571] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:34

/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# 70 30 Split

In [ ]:
wine_7030     = all_splits["7030"]["wine"]
logreg_results += log_reg_data(wine_7030, "wine", "7030")

[wine 7030] LR-L2
accuracy: 0.9949
precision: 0.9876
recall: 0.9917
f1_score: 0.9896
auc_roc: 0.9938
[W] [00:25:15.189229] QWL-QN line search failed (code 3); stopping at the last valid step
[wine 7030] LR-L1
accuracy: 0.9949
precision: 0.9876
recall: 0.9917
f1_score: 0.9896
auc_roc: 0.9938
[wine 7030] LR-ElasticNet
accuracy: 0.9949
precision: 0.9876
recall: 0.9917
f1_score: 0.9896
auc_roc: 0.9938


In [ ]:
customer_7030 = all_splits["7030"]["customer"]
logreg_results += log_reg_data(customer_7030, "customer", "7030")


[customer 7030] LR-L2
accuracy: 0.6871
precision: 0.7722
recall: 0.6061
f1_score: 0.6791
auc_roc: 0.6954
[customer 7030] LR-L1
accuracy: 0.6854
precision: 0.7778
recall: 0.5939
f1_score: 0.6735
auc_roc: 0.6948
[customer 7030] LR-ElasticNet
accuracy: 0.6854
precision: 0.7778
recall: 0.5939
f1_score: 0.6735
auc_roc: 0.6948


In [ ]:
cup98_7030    = all_splits["7030"]["cup98"]
logreg_results += log_reg_data(cup98_7030, "cup98", "7030")

[W] [00:25:17.038374] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:17.478359] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:17.915949] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:18.339102] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:18.929939] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:19.346207] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:19.695132] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:20.216962] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:20.526085] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:20.725530] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:21.007294] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25

# 50 50 Split

In [ ]:
wine_5050     = all_splits["5050"]["wine"]
logreg_results += log_reg_data(wine_5050, "wine", "5050")

[wine 5050] LR-L2
accuracy: 0.9923
precision: 0.9790
recall: 0.9900
f1_score: 0.9845
auc_roc: 0.9915
[W] [00:25:48.155661] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:25:48.408059] QWL-QN line search failed (code 3); stopping at the last valid step
[wine 5050] LR-L1
accuracy: 0.9926
precision: 0.9802
recall: 0.9900
f1_score: 0.9851
auc_roc: 0.9917
[wine 5050] LR-ElasticNet
accuracy: 0.9923
precision: 0.9802
recall: 0.9888
f1_score: 0.9844
auc_roc: 0.9911


In [ ]:
customer_5050 = all_splits["5050"]["customer"]
logreg_results += log_reg_data(customer_5050, "customer", "5050")

[customer 5050] LR-L2
accuracy: 0.6991
precision: 0.7839
recall: 0.6200
f1_score: 0.6924
auc_roc: 0.7072
[customer 5050] LR-L1
accuracy: 0.6971
precision: 0.7790
recall: 0.6218
f1_score: 0.6916
auc_roc: 0.7048
[customer 5050] LR-ElasticNet
accuracy: 0.6971
precision: 0.7790
recall: 0.6218
f1_score: 0.6916
auc_roc: 0.7048


In [ ]:
cup98_5050    = all_splits["5050"]["cup98"]
logreg_results += log_reg_data(cup98_5050, "cup98", "5050")

[W] [00:25:50.342808] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:50.807618] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:51.285528] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:51.687920] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:52.233875] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:52.590262] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:53.183038] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:53.541308] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:53.870408] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:54.066988] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25:54.185019] L-BFGS line search failed (code 3); stopping at the last valid step
[W] [00:25

/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[cup98 5050] LR-L1
accuracy: 0.9492
precision: 0.0000
recall: 0.0000
f1_score: 0.0000
auc_roc: 0.5000
[W] [00:26:08.827878] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:10.167071] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:12.043762] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:13.315690] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:14.158114] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:16.192873] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:17.937466] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:19.032290] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:20.015241] QWL-QN line search failed (code 3); stopping at the last valid step
[W] [00:26:20.144024] QWL-QN line search failed (code 3); stopping at the last valid ste

/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
import pandas as pd

df = pd.DataFrame(logreg_results)
df.to_csv("logreg_results_all.csv", index=False)
df


,dataset,split,model,penalty,lambda,accuracy,precision,recall,f1_score,auc_roc
0,wine,3070,LR,l2,1.000000e+00,0.991866,0.976253,0.991071,0.983607,0.991599
1,wine,3070,LR,l1,1.000000e+01,0.991207,0.974517,0.990179,0.982285,0.990861
2,wine,3070,LR,elasticnet,1.000000e+00,0.992086,0.977954,0.990179,0.984028,0.991444
3,customer,3070,LR,l2,1.000000e-01,0.693617,0.772581,0.622078,0.689209,0.700883
4,customer,3070,LR,l1,1.000000e-07,0.692908,0.772213,0.620779,0.688265,0.700233
5,customer,3070,LR,elasticnet,1.000000e-07,0.692908,0.772213,0.620779,0.688265,0.700233
6,cup98,3070,LR,l2,1.000000e+02,0.623447,0.067982,0.504866,0.119829,0.567327
7,cup98,3070,LR,l1,1.000000e+04,0.949229,0.000000,0.000000,0.000000,0.500000
8,cup98,3070,LR,elasticnet,1.000000e+04,0.949229,0.000000,0.000000,0.000000,0.500000
9,wine,7030,LR,l2,1.000000e-07,0.994872,0.987552,0.991667,0.989605,0.993793


/home/james/miniconda3/envs/rapids-24/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
